# Tutorial 19: Boltz-2 Structure Embeddings

This tutorial demonstrates how to extract **structural embeddings**
from Boltz-2, a biomolecular interaction prediction model.

### What this tutorial covers

1. **Boltz-2 overview** -- what it is and what representations it produces
2. **Embed proteins** with single, pairwise, and combined representations
3. **Compare with ESM-2** -- sequence vs structure embeddings
4. **Visualize** with embpy.pl

### Boltz-2 representations

| Model key | Output | Dimensions | Description |
|---|---|---|---|
| `boltz2` | Single | ~384 | Per-residue representation from pairformer trunk, pooled |
| `boltz2_pairwise` | Pairwise | ~128 | Inter-residue interaction features, pooled |
| `boltz2_both` | Both | ~512 | Concatenation of single + pairwise |

### Requirements

```bash
pip install boltz[cuda]
```

First run will download the Boltz-2 checkpoint (~2GB) and CCD dictionary (~150MB).

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import torch
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import embpy.pl as epl

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.1)
%matplotlib inline

try:
    import boltz
    print(f'Boltz version: {boltz.__version__}')
    BOLTZ_AVAILABLE = True
except ImportError:
    print('Boltz not installed. Run: pip install boltz[cuda]')
    BOLTZ_AVAILABLE = False

---
## 1. Initialize BioEmbedder

In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device='auto')
print(f'Device: {embedder.device}')

available = embedder.list_available_models(category='protein')
boltz_models = [m for m in available if 'boltz' in m]
print(f'Boltz models available: {boltz_models}')

---
## 2. Embed proteins with Boltz-2

We embed 5 well-known proteins using all three Boltz-2 output types.

In [ ]:
PROTEINS = ['TP53', 'BRCA1', 'EGFR', 'KRAS', 'MYC']

if BOLTZ_AVAILABLE and boltz_models:
    boltz_single = {}
    boltz_pairwise = {}
    boltz_both = {}

    for gene in PROTEINS:
        try:
            emb_s = embedder.embed_protein(gene, model='boltz2', pooling_strategy='mean')
            boltz_single[gene] = emb_s
            print(f'  {gene:8s}  single={emb_s.shape[0]}')
        except Exception as e:
            print(f'  {gene:8s}  single FAILED: {e}')

        try:
            emb_z = embedder.embed_protein(gene, model='boltz2_pairwise', pooling_strategy='mean')
            boltz_pairwise[gene] = emb_z
            print(f'  {gene:8s}  pairwise={emb_z.shape[0]}')
        except Exception as e:
            print(f'  {gene:8s}  pairwise FAILED: {e}')

        try:
            emb_b = embedder.embed_protein(gene, model='boltz2_both', pooling_strategy='mean')
            boltz_both[gene] = emb_b
            print(f'  {gene:8s}  both={emb_b.shape[0]}')
        except Exception as e:
            print(f'  {gene:8s}  both FAILED: {e}')
else:
    print('Boltz-2 not available. Skipping embedding.')

---
## 3. Compare with ESM-2

In [ ]:
esm_embs = {}
for gene in PROTEINS:
    try:
        emb = embedder.embed_protein(gene, model='esm2_650M', pooling_strategy='mean')
        esm_embs[gene] = emb
        print(f'  {gene:8s}  ESM-2 dim={emb.shape[0]}')
    except Exception as e:
        print(f'  {gene:8s}  ESM-2 FAILED: {e}')

In [ ]:
if BOLTZ_AVAILABLE and boltz_single and esm_embs:
    common = [g for g in PROTEINS if g in boltz_single and g in esm_embs]

    from embpy.pp import reduce_embeddings

    adata = ad.AnnData(
        obs=pd.DataFrame({'gene': common}, index=pd.Index(common)),
    )
    adata.obsm['X_boltz2_single'] = np.stack([boltz_single[g] for g in common]).astype(np.float32)
    adata.obsm['X_esm2'] = np.stack([esm_embs[g] for g in common]).astype(np.float32)

    if boltz_pairwise:
        common_pw = [g for g in common if g in boltz_pairwise]
        if len(common_pw) == len(common):
            adata.obsm['X_boltz2_pairwise'] = np.stack(
                [boltz_pairwise[g] for g in common]
            ).astype(np.float32)

    print(adata)
    for k in adata.obsm:
        print(f'  {k}: {adata.obsm[k].shape}')

---
## 4. Visualizations

In [ ]:
if BOLTZ_AVAILABLE and 'X_boltz2_single' in getattr(adata, 'obsm', {}):
    epl.plot_similarity_heatmap(
        adata=adata, obsm_key='X_boltz2_single', metric='cosine',
        labels=common, title='Boltz-2 single: cosine similarity',
        annot=True, fmt='.2f',
    )
    plt.show()

    epl.plot_similarity_heatmap(
        adata=adata, obsm_key='X_esm2', metric='cosine',
        labels=common, title='ESM-2: cosine similarity',
        annot=True, fmt='.2f',
    )
    plt.show()

    epl.cross_embedding_correlation(
        adata, obsm_key_a='X_boltz2_single', obsm_key_b='X_esm2',
        method='pearson',
    )
    plt.show()
else:
    print('Boltz-2 embeddings not available for visualization.')

---
## Summary

### When to use Boltz-2 vs ESM-2

| Aspect | ESM-2 | Boltz-2 |
|---|---|---|
| Speed | Fast (~0.1s per protein) | Slower (~10-30s, structure prediction) |
| GPU memory | ~2GB (650M) | ~4GB |
| Input | Sequence only | Sequence + optional MSA |
| What it captures | Evolutionary patterns | 3D structural context |
| Pairwise features | No | Yes (inter-residue interactions) |
| Best for | General protein embeddings | Structure-aware tasks, PPI |

### API

```python
# Single representation (default)
emb = embedder.embed_protein('TP53', model='boltz2')

# Pairwise representation
emb = embedder.embed_protein('TP53', model='boltz2_pairwise')

# Both concatenated
emb = embedder.embed_protein('TP53', model='boltz2_both')
```